In [ ]:
class RCNN(nn.Module):
    """Klasszikus RCNN modell: backbone + SVM osztályozók + Ridge regresszorok."""
    def __init__(self, num_classes=20):
        super().__init__()
        self.num_classes = num_classes
        self.feature_extractor = FeatureExtractor()
        self.svm_classifiers = {}   # osztály_index -> LinearSVC
        self.ridge_regressors = {}  # osztály_index -> Ridge
        self.scaler = StandardScaler()
        self.class_names = VOC_CLASSES

    def extract_features(self, image, regions, batch_size=64):
        return extract_region_features(image, regions, batch_size)

    def fit_svm(self, features, labels):
        """SVM osztályozók tanítása osztályonként (one-vs-rest)."""
        features_scaled = self.scaler.fit_transform(features)
        for cls_idx in range(self.num_classes):
            y = (labels == cls_idx).astype(int)
            if y.sum() == 0:
                continue
            svm = LinearSVC(C=0.1, max_iter=5000, dual='auto', random_state=42)
            svm.fit(features_scaled, y)
            self.svm_classifiers[cls_idx] = svm
        print(f"SVM-ek tanítva {len(self.svm_classifiers)} osztályra.")

    def fit_regressors(self, features, labels, gt_boxes, proposal_boxes):
        """Bbox regresszorok tanítása osztályonként."""
        features_scaled = self.scaler.transform(features)
        for cls_idx in range(self.num_classes):
            mask = labels == cls_idx
            if mask.sum() < 10:
                continue
            X = features_scaled[mask]
            # Cél: [tx, ty, tw, th] transzformációk
            targets = []
            for j in np.where(mask)[0]:
                gt = gt_boxes[j]
                prop = proposal_boxes[j]
                tx = (gt[0] - prop[0]) / prop[2]
                ty = (gt[1] - prop[1]) / prop[3]
                tw = np.log(max(gt[2], 1e-6) / max(prop[2], 1e-6))
                th = np.log(max(gt[3], 1e-6) / max(prop[3], 1e-6))
                targets.append([tx, ty, tw, th])
            targets = np.array(targets)
            ridge = Ridge(alpha=1000)
            ridge.fit(X, targets)
            self.ridge_regressors[cls_idx] = ridge
        print(f"Regresszorok tanítva {len(self.ridge_regressors)} osztályra.")

    def predict(self, image, regions):
        """Predikció: osztály + bbox finomítás minden régióra."""
        if len(regions) == 0:
            return [], [], []
        feats = self.extract_features(image, regions)
        feats_scaled = self.scaler.transform(feats)
        scores = np.zeros((len(regions), self.num_classes))
        refined_boxes = []
        for cls_idx in range(self.num_classes):
            if cls_idx in self.svm_classifiers:
                # Confidence score
                decision = self.svm_classifiers[cls_idx].decision_function(feats_scaled)
                scores[:, cls_idx] = decision
                # Bbox finomítás
                if cls_idx in self.ridge_regressors:
                    deltas = self.ridge_regressors[cls_idx].predict(feats_scaled)
                    for j, (x, y, w, h) in enumerate(regions):
                        tx, ty, tw, th = deltas[j]
                        cx = x + w/2
                        cy = y + h/2
                        new_cx = cx + tx * w
                        new_cy = cy + ty * h
                        new_w = w * np.exp(tw)
                        new_h = h * np.exp(th)
                        refined_boxes.append((new_cx - new_w/2, new_cy - new_h/2,
                                              new_cx + new_w/2, new_cy + new_h/2))
        return scores, refined_boxes

print("RCNN osztály definiálva.")

In [ ]:
# Kép transzformáció a ResNet-hez
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_region_features(image, regions, batch_size=64):
    """Régiók kivágása, átméretezése és feature kinyerése batch-elve."""
    features_list = []
    for i in range(0, len(regions), batch_size):
        batch_regions = regions[i:i+batch_size]
        batch_tensors = []
        for (x, y, w, h) in batch_regions:
            crop = image.crop((x, y, x+w, y+h))
            crop_tensor = transform(crop)
            batch_tensors.append(crop_tensor)
        batch = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            feats = feature_extractor(batch)
        features_list.append(feats.cpu().numpy())
    return np.vstack(features_list) if features_list else np.array([])

print("Feature kinyerő függvények definiálva.")

In [ ]:
# Feature extractor: ResNet50 utolsó osztályozó réteg nélkül
class FeatureExtractor(nn.Module):
    """ResNet50 backbone, az avgpool előtti feature map-eket adja vissza."""
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Távolítsuk el az utolsó fc és avgpool réteget
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.output_dim = 2048

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

# Eszköz beállítása
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
feature_extractor = FeatureExtractor().to(device)
feature_extractor.eval()
print(f"Feature extractor létrehozva. Kimeneti dimenzió: {feature_extractor.output_dim}")
print(f"Eszköz: {device}")

## 2. RCNN háló létrehozása

ResNet50 backbone ImageNet előtanítással, régiók átméretezése 224x224-re, SVM osztályozó és Ridge regresszor osztályonként.

In [ ]:
# Vizualizáció: minta képek annotációkkal és statisztikák

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 3 minta kép bboxokkal
sample_ids = random.sample(train_images[:500], 3)
for i, img_id in enumerate(sample_ids):
    img_path = images_dir / f"{img_id}.jpg"
    img = Image.open(img_path)
    xml_path = annotations_dir / f"{img_id}.xml"
    objects, _ = parse_voc_xml(xml_path)

    axes[0, i].imshow(img)
    for cls, bbox in objects:
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                                   linewidth=2, edgecolor='red', facecolor='none')
        axes[0, i].add_patch(rect)
        axes[0, i].text(bbox[0], bbox[1]-5, cls, color='red', fontsize=8)
    axes[0, i].set_title(f"{img_id} ({len(objects)} objektum)")
    axes[0, i].axis('off')

# Osztályeloszlás hisztogram
axes[1, 0].bar(range(len(VOC_CLASSES)), [class_counts[c] for c in VOC_CLASSES])
axes[1, 0].set_xticks(range(len(VOC_CLASSES)))
axes[1, 0].set_xticklabels(VOC_CLASSES, rotation=90, fontsize=7)
axes[1, 0].set_title("Osztályeloszlás")

# Bbox terület hisztogram
axes[1, 1].hist(bbox_areas, bins=50, edgecolor='black')
axes[1, 1].set_title("Bbox területek eloszlása")
axes[1, 1].set_xlabel("Terület (px²)")

# Bbox arány hisztogram
axes[1, 2].hist(bbox_ratios, bins=50, edgecolor='black')
axes[1, 2].set_title("Bbox szélesség/magasság arány")
axes[1, 2].set_xlabel("w/h arány")

plt.tight_layout()
plt.savefig('dataset_exploration.png', dpi=100)
plt.show()
print("Adathalmaz feltérképezés kész.")

In [ ]:
# Osztályok listája
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def parse_voc_xml(xml_path):
    """VOC XML annotáció parse-olása. Visszaadja: [(osztály, [xmin,ymin,xmax,ymax]), ...]"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append((name, [xmin, ymin, xmax, ymax]))
    return objects, (width, height)

# Osztályeloszlás vizsgálata
class_counts = defaultdict(int)
bbox_areas = []
bbox_ratios = []
image_sizes = []

for img_id in train_images[:500]:  # első 500 kép a gyors elemzéshez
    xml_path = annotations_dir / f"{img_id}.xml"
    if xml_path.exists():
        objects, (w, h) = parse_voc_xml(xml_path)
        image_sizes.append((w, h))
        for cls, bbox in objects:
            class_counts[cls] += 1
            area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            ratio = (bbox[2] - bbox[0]) / max(bbox[3] - bbox[1], 1)
            bbox_areas.append(area)
            bbox_ratios.append(ratio)

print("=== Osztályeloszlás (első 500 train kép) ===")
for cls in VOC_CLASSES:
    print(f"  {cls}: {class_counts[cls]}")

print(f"\nBbox terület: min={min(bbox_areas):.0f}, max={max(bbox_areas):.0f}, mean={np.mean(bbox_areas):.0f}")
print(f"Bbox arány (w/h): mean={np.mean(bbox_ratios):.2f}")
print(f"Képméret: min={min(image_sizes)}, max={max(image_sizes)}")

In [ ]:
# Pascal VOC 2012 letöltése
path = kagglehub.dataset_download("gopalbhattrai/pascal-voc-2012-dataset")
print(f"Adathalmaz elérési út: {path}")

# Mappaszerkezet feltárása
voc_root = Path(path) / "VOC2012"
annotations_dir = voc_root / "Annotations"
images_dir = voc_root / "JPEGImages"
imgset_dir = voc_root / "ImageSets" / "Main"

print(f"Annotations: {annotations_dir.exists()}")
print(f"Images: {images_dir.exists()}")

# Képek listája train/val splitből
train_images = []
for class_file in sorted(imgset_dir.glob("*_train.txt")):
    with open(class_file) as f:
        train_images.extend([line.strip().split()[0] for line in f if line.strip()])
train_images = list(set(train_images))

val_images = []
for class_file in sorted(imgset_dir.glob("*_val.txt")):
    with open(class_file) as f:
        val_images.extend([line.strip().split()[0] for line in f if line.strip()])
val_images = list(set(val_images))

print(f"Train képek: {len(train_images)}")
print(f"Val képek: {len(val_images)}")

# RCNN Object Detection - Pascal VOC 2012

Klasszikus RCNN (Girshick et al. 2014) implementáció PyTorch és scikit-learn használatával.
ResNet50 backbone, Selective Search régió javaslatok, SVM osztályozás.

## 0. Környezet beállítása

In [ ]:
# GPU ellenőrzés
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA elérhető: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Függőségek telepítése
!pip install -q kagglehub opencv-python scikit-learn matplotlib pillow torch torchvision

import kagglehub
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.svm import LinearSVC
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import xml.etree.ElementTree as ET
import os
import random
import pickle
from collections import defaultdict
from pathlib import Path

print("Minden függőség betöltve.")

## 1. Adathalmaz feltérképezése

Pascal VOC 2012 letöltése kagglehub segítségével, osztályeloszlás, képméretek és bbox statisztikák vizsgálata.